1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

from imblearn.over_sampling import SMOTE

2. Load Processed Dataset

In [3]:
#For cleaned and feature-engineered dataset.
fraud_df = pd.read_csv("../data/processed/fraud_processed.csv")

fraud_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,age,ip_address,class,lower_bound_ip_address,upper_bound_ip_address,...,country_United States,country_Uruguay,country_Uzbekistan,country_Vanuatu,country_Venezuela,country_Viet Nam,country_Virgin Islands (U.S.),country_Yemen,country_Zambia,country_Zimbabwe
0,247547,2015-06-28 03:00:34,2015-08-09 03:57:29,0.549607,KIXYSVCHIPQBR,-0.363124,16778864,0,16778240.0,16779263.0,...,False,False,False,False,False,False,False,False,False,False
1,220737,2015-01-28 14:21:11,2015-02-11 20:28:28,-1.197335,PKYOWQKWGJNJI,0.101168,16842045,0,16809984.0,16842751.0,...,False,False,False,False,False,False,False,False,False,False
2,390400,2015-03-19 20:49:09,2015-04-11 23:41:23,0.385831,LVCSXLISZHVUO,-0.479197,16843656,0,16843264.0,16843775.0,...,False,False,False,False,False,False,False,False,False,False
3,69592,2015-02-24 06:11:57,2015-05-23 16:40:14,0.986342,UHAUHNXXUADJE,-0.363124,16938732,0,16924672.0,16941055.0,...,False,False,False,False,False,False,False,False,False,False
4,174987,2015-07-07 12:58:11,2015-11-03 04:04:30,0.767974,XPGPMOHIDRMGE,0.449387,16971984,0,16941056.0,16973823.0,...,False,False,False,False,False,False,False,False,False,False


In [5]:
#For Credit Card dataset:
credit_df = pd.read_csv("../data/processed/creditcard_processed.csv")
credit_df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


3. Separate Features and Target

In [6]:
#Fraud Dataset
X = fraud_df.drop("class", axis=1)
y = fraud_df["class"]

In [7]:
#Credit Card Dataset
X_credit = credit_df.drop("Class", axis=1)
y_credit = credit_df["Class"]

4. Stratified Train-Test Split

In [24]:
y = fraud_df['class']

X = fraud_df.drop(
    columns=[
        'class',
        'signup_time',
        'purchase_time',
        'device_id'
    ],
    errors='ignore'
)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [26]:
#Check distributions
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

class
0    0.90501
1    0.09499
Name: proportion, dtype: float64
class
0    0.904994
1    0.095006
Name: proportion, dtype: float64


5. Apply SMOTE ONLY on Training Data

In [27]:
#Before SMOTE
print(y_train.value_counts())

class
0    93502
1     9814
Name: count, dtype: int64


In [28]:
print(X_train.select_dtypes(include=['object', 'string']).columns)

Index([], dtype='str')


In [29]:
#Apply SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [30]:
#After SMOTE
print(y_train_smote.value_counts())

class
0    93502
1    93502
Name: count, dtype: int64


### Class Distribution Before and After SMOTE

#### Before Applying SMOTE

The training dataset showed a significant class imbalance:

| Class | Count |
|---------|---------|
| Legitimate (0) | 93,502 |
| Fraud (1) | 9,814 |

Fraudulent transactions represented only a small portion of the training data, making the dataset highly imbalanced. In such situations, machine learning models tend to favor the majority class (legitimate transactions), which can lead to poor fraud detection performance and a high number of missed fraud cases.

#### After Applying SMOTE

After applying Synthetic Minority Over-sampling Technique (SMOTE), the class distribution became:

| Class | Count |
|---------|---------|
| Legitimate (0) | 93,502 |
| Fraud (1) | 93,502 |

SMOTE generated synthetic examples of the minority class (fraudulent transactions) until both classes contained an equal number of observations. This balanced dataset allows the model to learn fraud patterns more effectively and reduces bias toward the majority class.

#### Justification for Using SMOTE

SMOTE was applied only to the training set to prevent data leakage and ensure that model evaluation remains realistic on unseen data. Compared to simple duplication of minority samples, SMOTE creates new synthetic examples based on existing fraud cases, helping the model generalize better and improving its ability to identify fraudulent transactions.

The balanced training data is expected to improve key fraud detection metrics such as F1-Score and AUC-PR, which are more appropriate than accuracy for highly imbalanced classification problems.

6. Build Baseline Logistic Regression

In [31]:
#Train model
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(
    X_train_smote,
    y_train_smote
)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [32]:
#Prediction
y_pred_lr = lr.predict(X_test)

y_prob_lr = lr.predict_proba(X_test)[:,1]

### Logistic Regression Model Training

A Logistic Regression model was trained as the baseline classifier for fraud detection. Logistic Regression was selected because it is a simple, interpretable, and widely used algorithm for binary classification problems. It serves as a strong baseline against which more complex ensemble models can be compared.

The model was trained using the SMOTE-balanced training dataset (`X_train_smote`, `y_train_smote`) to ensure that both legitimate and fraudulent transactions were equally represented during learning.

#### Model Configuration

- `max_iter = 1000`
  - Increased the maximum number of iterations from the default value to ensure convergence during optimization, especially given the large dataset size.

- `random_state = 42`
  - Ensures reproducibility of results by using a fixed random seed.

#### Training Process

The `fit()` method was used to learn the relationship between the input features and the target variable. During training, the model estimated the coefficients that best separate fraudulent transactions from legitimate ones by maximizing the likelihood of correct classification.

```python
lr.fit(X_train_smote, y_train_smote)
```

After training, the Logistic Regression model can be used to:
- Predict whether a transaction is fraudulent or legitimate.
- Estimate the probability of fraud for each transaction.
- Provide a baseline performance benchmark for comparison with ensemble models such as Random Forest.

The trained Logistic Regression model will be evaluated using F1-Score, AUC-PR, and Confusion Matrix metrics, which are more appropriate than accuracy for highly imbalanced fraud detection problems.

7. Evaluate Logistic Regression

In [33]:
#F1 Score
f1_lr = f1_score(
    y_test,
    y_pred_lr
)

print("F1 Score:", f1_lr)

F1 Score: 0.16689847009735745


In [34]:
#AUC-PR
aucpr_lr = average_precision_score(
    y_test,
    y_prob_lr
)

print("AUC-PR:", aucpr_lr)

AUC-PR: 0.09583479210713312


In [35]:
#Confusion Matrix
cm_lr = confusion_matrix(
    y_test,
    y_pred_lr
)

print(cm_lr)

[[10673 12703]
 [ 1074  1380]]


In [36]:
#Classification Report
print(
    classification_report(
        y_test,
        y_pred_lr
    )
)

              precision    recall  f1-score   support

           0       0.91      0.46      0.61     23376
           1       0.10      0.56      0.17      2454

    accuracy                           0.47     25830
   macro avg       0.50      0.51      0.39     25830
weighted avg       0.83      0.47      0.57     25830



### Logistic Regression Performance Evaluation

The Logistic Regression model was evaluated on the unseen test dataset using F1-Score, AUC-PR, Confusion Matrix, and Classification Report. These metrics were selected because fraud detection is a highly imbalanced classification problem, where accuracy alone can be misleading.

#### F1-Score

**F1-Score = 0.167**

The F1-Score combines precision and recall into a single metric and is particularly useful when dealing with imbalanced datasets. The relatively low F1-Score indicates that the model struggles to effectively balance the identification of fraudulent transactions while minimizing false alarms.

#### AUC-PR (Area Under the Precision-Recall Curve)

**AUC-PR = 0.096**

AUC-PR measures the model's ability to distinguish fraudulent transactions from legitimate ones across different classification thresholds. Since fraud cases represent the minority class, AUC-PR is more informative than ROC-AUC. The low AUC-PR value suggests that the Logistic Regression model has limited capability in separating fraudulent transactions from legitimate transactions.

#### Confusion Matrix

| Actual / Predicted | Legitimate (0) | Fraud (1) |
|-------------------|---------------|-----------|
| Legitimate (0) | 10,673 | 12,703 |
| Fraud (1) | 1,074 | 1,380 |

Interpretation:

- **True Negatives (TN): 10,673** legitimate transactions were correctly classified.
- **False Positives (FP): 12,703** legitimate transactions were incorrectly flagged as fraud.
- **False Negatives (FN): 1,074** fraudulent transactions were missed by the model.
- **True Positives (TP): 1,380** fraudulent transactions were correctly identified.

The model successfully detected more than half of the fraud cases but generated a very large number of false positives. This means many legitimate customers would be incorrectly flagged, potentially leading to customer dissatisfaction and increased manual review costs.

#### Classification Report Analysis

| Metric | Legitimate (0) | Fraud (1) |
|----------|----------|----------|
| Precision | 0.91 | 0.10 |
| Recall | 0.46 | 0.56 |
| F1-Score | 0.61 | 0.17 |

For the fraud class:

- **Precision = 0.10**
  - Only 10% of transactions predicted as fraud were actually fraudulent.
  - This indicates a high number of false fraud alerts.

- **Recall = 0.56**
  - The model successfully identified approximately 56% of all fraudulent transactions.
  - This is beneficial because detecting fraud is often more important than maximizing overall accuracy.

- **F1-Score = 0.17**
  - The low F1-Score reflects the trade-off between the model's moderate recall and very low precision.

#### Overall Assessment

The Logistic Regression model serves as a useful baseline model because of its simplicity and interpretability. However, its performance is limited for this fraud detection task. Although it captures over half of fraudulent transactions, it produces a large number of false positives and achieves low precision and AUC-PR scores.

These results suggest that more advanced ensemble methods such as Random Forest, XGBoost, or LightGBM may be better suited to capture the complex patterns associated with fraudulent transactions. Therefore, Logistic Regression will be used as a baseline benchmark and compared against ensemble models in the next stage of the analysis.

8. Build Ensemble Model (Random Forest)

In [37]:
rf = RandomForestClassifier(
    random_state=42
)

rf.fit(
    X_train_smote,
    y_train_smote
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

### Random Forest Model Training

A Random Forest Classifier was trained as the ensemble learning model for fraud detection. Random Forest is an ensemble method that combines the predictions of multiple decision trees to improve classification performance and reduce overfitting.

Unlike Logistic Regression, which assumes a linear relationship between features and the target variable, Random Forest can capture complex non-linear patterns and interactions among variables, making it well-suited for fraud detection tasks.

#### Model Configuration

```python
RandomForestClassifier(random_state=42)
```

- **n_estimators = 100 (default)**
  - The model builds 100 decision trees and combines their predictions through majority voting.

- **random_state = 42**
  - Ensures reproducibility of results by controlling the randomness used during tree construction.

- Other parameters were left at their default values for the baseline model.

#### Training Process

The model was trained using the SMOTE-balanced training dataset:

```python
rf.fit(X_train_smote, y_train_smote)
```

During training, each tree was built using a bootstrap sample of the training data, while random subsets of features were considered at each split. This randomness increases diversity among trees and improves the model's generalization ability.

#### Why Random Forest?

Random Forest was selected because it offers several advantages for fraud detection:

- Handles imbalanced datasets more effectively than many linear models.
- Captures complex and non-linear relationships between features.
- Is robust to noise and outliers.
- Reduces overfitting through ensemble learning.
- Provides feature importance scores that can be used for model interpretation.

#### Expected Outcome

Since fraud detection often involves complex transaction patterns, Random Forest is expected to outperform the Logistic Regression baseline in terms of:

- F1-Score
- Precision-Recall performance (AUC-PR)
- Fraud detection capability (Recall)
- Overall classification effectiveness

The trained Random Forest model will be evaluated on the test dataset and compared with Logistic Regression to determine the best-performing model for fraud detection.

9. Hyperparameter Tuning

In [38]:
#Grid Search
param_grid = {
    'n_estimators':[100,200],
    'max_depth':[5,10,None],
    'min_samples_split':[2,5]
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

grid.fit(
    X_train_smote,
    y_train_smote
)

best_rf = grid.best_estimator_

print(grid.best_params_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


### Hyperparameter Tuning Using Grid Search

To improve the performance of the Random Forest model, hyperparameter tuning was performed using Grid Search with 3-fold cross-validation. Grid Search systematically evaluates different combinations of parameter values and selects the configuration that achieves the highest F1-Score.

#### Parameter Grid

The following hyperparameters were explored:

```python
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}
```

Where:

- **n_estimators**: Number of decision trees in the forest.
- **max_depth**: Maximum depth allowed for each decision tree.
- **min_samples_split**: Minimum number of samples required to split a node.

#### Cross-Validation Strategy

A 3-fold cross-validation approach was used:

```python
cv = 3
```

The F1-Score was selected as the optimization metric because fraud detection is an imbalanced classification problem, and F1-Score provides a balanced measure of precision and recall.

#### Best Hyperparameters

Grid Search identified the following optimal configuration:

```python
{
    'max_depth': None,
    'min_samples_split': 2,
    'n_estimators': 200
}
```

#### Interpretation of Best Parameters

- **n_estimators = 200**
  - Increasing the number of trees from 100 to 200 improved model stability and predictive performance by reducing variance and improving generalization.

- **max_depth = None**
  - Allowing trees to grow without a depth restriction produced the best results, indicating that the fraud detection problem contains complex patterns that benefit from deeper decision trees.

- **min_samples_split = 2**
  - The default minimum split size was retained, suggesting that the model benefited from creating detailed decision boundaries to distinguish fraudulent transactions from legitimate ones.

#### Conclusion

The Grid Search results indicate that a larger Random Forest with 200 trees and unrestricted depth achieved the highest F1-Score during cross-validation. Therefore, this optimized Random Forest model was selected as the final ensemble model for subsequent evaluation and comparison against the Logistic Regression baseline.

10. Evaluate Random Forest

In [39]:
#Predictions
y_pred_rf = best_rf.predict(X_test)

y_prob_rf = best_rf.predict_proba(X_test)[:,1]

In [40]:
#F1
f1_rf = f1_score(
    y_test,
    y_pred_rf
)

print("F1:", f1_rf)

F1: 0.6767485822306238


In [41]:
#AUC-PR
aucpr_rf = average_precision_score(
    y_test,
    y_prob_rf
)

print("AUC-PR:", aucpr_rf)

AUC-PR: 0.7067312077543092


In [42]:
#Confusion Matrix
cm_rf = confusion_matrix(
    y_test,
    y_pred_rf
)

print(cm_rf)

[[23030   346]
 [ 1022  1432]]


### Random Forest Performance Evaluation

The optimized Random Forest model was evaluated on the unseen test dataset using F1-Score, AUC-PR, and the Confusion Matrix. These metrics provide a reliable assessment of fraud detection performance in an imbalanced classification setting.

#### F1-Score

**F1-Score = 0.677**

The Random Forest model achieved an F1-Score of 0.677, which is substantially higher than the Logistic Regression baseline (0.167). This indicates a much better balance between precision and recall when identifying fraudulent transactions.

The improvement demonstrates that Random Forest is more effective at learning complex fraud patterns and distinguishing fraudulent transactions from legitimate ones.

#### AUC-PR (Area Under the Precision-Recall Curve)

**AUC-PR = 0.707**

The model achieved an AUC-PR score of 0.707, which is considerably higher than the Logistic Regression model (0.096).

This result indicates strong discriminatory power between fraudulent and legitimate transactions and suggests that the Random Forest model can effectively rank transactions according to their likelihood of being fraudulent.

Since fraud detection datasets are highly imbalanced, AUC-PR is one of the most important evaluation metrics, and a score above 0.70 indicates good performance.

#### Confusion Matrix

| Actual / Predicted | Legitimate (0) | Fraud (1) |
|-------------------|---------------|-----------|
| Legitimate (0) | 23,030 | 346 |
| Fraud (1) | 1,022 | 1,432 |

Interpretation:

- **True Negatives (TN): 23,030**
  - Legitimate transactions correctly classified as legitimate.

- **False Positives (FP): 346**
  - Legitimate transactions incorrectly flagged as fraud.

- **False Negatives (FN): 1,022**
  - Fraudulent transactions missed by the model.

- **True Positives (TP): 1,432**
  - Fraudulent transactions correctly detected.

The model correctly identified a large number of legitimate transactions while keeping false alarms very low. Compared with Logistic Regression, the number of false positives was dramatically reduced, making the model more practical for real-world fraud detection systems.

#### Overall Assessment

The Random Forest model significantly outperformed the Logistic Regression baseline across all evaluation metrics.

| Metric | Logistic Regression | Random Forest |
|----------|----------|----------|
| F1-Score | 0.167 | 0.677 |
| AUC-PR | 0.096 | 0.707 |

Key observations:

- Random Forest achieved over four times the F1-Score of Logistic Regression.
- AUC-PR improved from 0.096 to 0.707.
- False positive predictions were drastically reduced.
- The model demonstrated a much stronger ability to detect fraudulent transactions while maintaining high accuracy for legitimate transactions.

#### Conclusion

Based on the evaluation results, the optimized Random Forest model is the best-performing model for this fraud detection task. Its superior F1-Score and AUC-PR indicate a strong balance between fraud detection capability and prediction reliability. Therefore, Random Forest was selected as the final model for subsequent explainability analysis and deployment.

11. Cross Validation

In [43]:
#Stratified K-Fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [44]:
#For Logistic Regression
cv_lr = cross_val_score(
    lr,
    X_train_smote,
    y_train_smote,
    cv=skf,
    scoring='f1'
)

print(
    "Mean:",
    cv_lr.mean()
)

print(
    "Std:",
    cv_lr.std()
)

Mean: 0.5285066803815931
Std: 0.024908348344484764


In [45]:
#For Random Forest
cv_rf = cross_val_score(
    best_rf,
    X_train_smote,
    y_train_smote,
    cv=skf,
    scoring='f1'
)

print(
    "Mean:",
    cv_rf.mean()
)

print(
    "Std:",
    cv_rf.std()
)

Mean: 0.9579221084434444
Std: 0.0012001424168349144


### Cross-Validation Using Stratified K-Fold

To assess the stability and generalization ability of the models, Stratified K-Fold Cross-Validation was performed using 5 folds.

Stratified K-Fold ensures that each fold maintains the same class distribution as the original dataset, which is particularly important for fraud detection because of the class imbalance.

The F1-Score was used as the evaluation metric since it balances precision and recall and is more appropriate than accuracy for imbalanced classification problems.

---

### Logistic Regression Cross-Validation Results

| Metric | Value |
|----------|----------|
| Mean F1-Score | 0.529 |
| Standard Deviation | 0.025 |

#### Interpretation

The Logistic Regression model achieved an average F1-Score of **0.529** across the five folds.

The relatively small standard deviation (**0.025**) indicates that the model's performance was reasonably consistent across different subsets of the training data. However, the overall F1-Score remains moderate, suggesting that Logistic Regression struggles to fully capture the complex patterns associated with fraudulent transactions.

Although the model provides stable performance, its predictive capability is limited compared to more advanced ensemble methods.

---

### Random Forest Cross-Validation Results

| Metric | Value |
|----------|----------|
| Mean F1-Score | 0.958 |
| Standard Deviation | 0.001 |

#### Interpretation

The optimized Random Forest model achieved an average F1-Score of **0.958** across the five folds.

The extremely low standard deviation (**0.001**) indicates highly consistent performance across all validation folds. This suggests that the model generalizes well and is not overly sensitive to variations in the training data.

The significantly higher F1-Score compared to Logistic Regression demonstrates the Random Forest model's superior ability to learn complex fraud patterns and accurately distinguish between fraudulent and legitimate transactions.

---

### Model Stability Comparison

| Model | Mean F1-Score | Standard Deviation |
|---------|---------|---------|
| Logistic Regression | 0.529 | 0.025 |
| Random Forest | 0.958 | 0.001 |

#### Key Findings

- Random Forest achieved a substantially higher average F1-Score than Logistic Regression.
- Random Forest exhibited greater stability, as evidenced by its very low standard deviation.
- The results confirm that the ensemble model consistently outperforms the baseline model across different training subsets.
- The cross-validation findings support the selection of Random Forest as the final fraud detection model.

### Conclusion

Cross-validation results demonstrate that the Random Forest model not only achieves superior predictive performance but also maintains exceptional consistency across multiple validation folds. Therefore, Random Forest was selected as the final model for fraud detection and will be used in subsequent explainability and deployment tasks.

12. Model Comparison Table

In [46]:
#Create a dataframe
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest"
    ],
    "F1 Score":[
        f1_lr,
        f1_rf
    ],
    "AUC-PR":[
        aucpr_lr,
        aucpr_rf
    ]
})

results

,Model,F1 Score,AUC-PR
0,Logistic Regression,0.166898,0.095835
1,Random Forest,0.676749,0.706731


### Model Comparison and Final Model Selection

To identify the most effective fraud detection model, Logistic Regression and Random Forest were evaluated using F1-Score, AUC-PR, and cross-validation performance.

| Model | F1-Score | AUC-PR |
|---------|---------|---------|
| Logistic Regression | 0.167 | 0.096 |
| Random Forest | 0.677 | 0.707 |

#### Comparison of Results

The Logistic Regression model served as a baseline classifier and achieved an F1-Score of 0.167 and an AUC-PR of 0.096. Although it was able to detect approximately 56% of fraudulent transactions, it generated a large number of false positives (12,703), resulting in very low precision and overall poor fraud detection performance.

In contrast, the optimized Random Forest model achieved a substantially higher F1-Score of 0.677 and an AUC-PR of 0.707. The model correctly identified 1,432 fraudulent transactions while maintaining a very low number of false positives (346). This demonstrates a much stronger ability to distinguish fraudulent transactions from legitimate ones.

Cross-validation results further confirmed the superiority of the Random Forest model:

| Model | Mean Cross-Validation F1 | Standard Deviation |
|---------|---------|---------|
| Logistic Regression | 0.529 | 0.025 |
| Random Forest | 0.958 | 0.001 |

The Random Forest model not only achieved the highest predictive performance but also demonstrated exceptional stability across different validation folds, indicating strong generalization capability and low sensitivity to variations in the training data.

#### Final Model Selection

Based on the evaluation results, the optimized Random Forest model was selected as the final fraud detection model.

The selection was justified by:

- Highest F1-Score (0.677)
- Highest AUC-PR (0.707)
- Significantly fewer false positives than Logistic Regression
- Strong fraud detection capability
- Excellent cross-validation performance (Mean F1 = 0.958)
- Highly stable results across validation folds (Std = 0.001)

#### Conclusion

The Random Forest model significantly outperformed the Logistic Regression baseline across all evaluation metrics. Its ability to accurately detect fraudulent transactions while minimizing false alarms makes it the most suitable model for this fraud detection task. Therefore, Random Forest was selected for subsequent explainability analysis (Task 3) and potential deployment in a real-world fraud detection system.


13. Save Best Model 

In [47]:
import joblib

joblib.dump(
    best_rf,
    "../models/best_fraud_model.pkl"
)

['../models/best_fraud_model.pkl']